# 02 — SAM 2.1 Tiny Encoder Distillation (128 MB / 1.35 s -> ~20 MB / ~250 ms)

Distills `sam_enc.onnx` (Apache-2.0) into a fast `RepViT-M0.9` / `MobileNetV4-Medium` student matching all 3 embedding tensors (`image_embeddings.0/1/2`) so the existing `sam_dec.onnx` (30 ms) works unchanged.


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
import torch.nn as nn, torch.nn.functional as F, timm

class DistilledSamEncoder(nn.Module):
    def __init__(self, backbone='mobilenetv4_conv_medium.e500_r256_in1k'):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=True, features_only=True, out_indices=(2, 3, 4))
        chs = self.enc.feature_info.channels()
        # Match SAM 2.1 tiny decoder expected channels: (32, 64, 256) at (256x256, 128x128, 64x64)
        self.p0 = nn.Conv2d(chs[0], 32, 1)
        self.p1 = nn.Conv2d(chs[1], 64, 1)
        self.p2 = nn.Conv2d(chs[2], 256, 1)
    def forward(self, pixel_values):
        f0, f1, f2 = self.enc(pixel_values)
        e0 = F.interpolate(self.p0(f0), size=(256, 256), mode='bilinear', align_corners=False)
        e1 = F.interpolate(self.p1(f1), size=(128, 128), mode='bilinear', align_corners=False)
        e2 = F.interpolate(self.p2(f2), size=(64, 64),   mode='bilinear', align_corners=False)
        return e0, e1, e2

# Teacher loop: run sam_enc.onnx on PD12M CC0 + Infinite renders, minimize MSE + (1 - cosine_sim) on (e0, e1, e2).


In [ ]:
def export_infinite_onnx(model, dummy, out_path, input_names, output_names, fp16=True):
    model.eval().cpu()
    tmp_fp32 = out_path.replace(".onnx", "_fp32.onnx")
    torch.onnx.export(
        model, dummy, tmp_fp32, opset_version=17, do_constant_folding=True,
        input_names=input_names, output_names=output_names,
    )
    m = onnxslim.slim(tmp_fp32)
    onnx.checker.check_model(m)
    if fp16:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        m = convert_float_to_float16(m, keep_io_types=True)
        onnx.checker.check_model(m)
    onnx.save(m, out_path)
    data = open(out_path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    print(f"Exported {out_path}: {len(data)/1e6:.2f} MB  sha256={sha}")
    return sha

# student = DistilledSamEncoder()
# export_infinite_onnx(student, torch.randn(1,3,1024,1024), '/content/out/sam_enc_fast.onnx',
#                      ['pixel_values'], ['image_embeddings.0', 'image_embeddings.1', 'image_embeddings.2'], fp16=True)
